# Part 5: The GIL - Very Important

Concurrency is about making progress on multiple tasks. Parallelism is about executing multiple tasks at the same time. Python supports both, but the best tool depends on whether work is CPU-bound, I/O-bound, or naturally asynchronous.

## Learning goals

By the end of this notebook, you should be able to:

- Explain the Global Interpreter Lock (GIL), its historical purpose, and its effect on threads.
- Choose threads for blocking I/O, processes for CPU-heavy pure Python work, and `asyncio` for cooperative I/O concurrency.
- Demonstrate race conditions and protect shared state with synchronization.
- Understand thread switching, scheduling, and why thread safety is not automatic.
- Describe the modern free-threaded CPython direction and its tradeoffs.

> Timing results depend on hardware and workload. Use the examples to understand behavior, not to claim universal benchmark numbers.

## 1. What is the GIL?

The **Global Interpreter Lock** is a mutual-exclusion lock in the traditional CPython runtime. It allows only one thread at a time to execute Python bytecode in a given interpreter process. Threads still exist and can run concurrently, but CPU-bound Python bytecode does not execute in true parallelism under the traditional GIL.

### Why CPython has had a GIL

The GIL simplified CPython's memory management and object model. In particular, it made reference-count updates and many internal operations safer without requiring fine-grained locks throughout the interpreter. It also preserved compatibility with a large ecosystem of C extensions that assumed simpler thread-safety rules.

The GIL is not a general-purpose application lock. It does not protect your logical invariants, and it does not eliminate race conditions between threads. A sequence such as `balance += amount` can still be interrupted between reading and writing shared state.

### CPU-bound versus I/O-bound work

- **CPU-bound:** spends most of its time executing Python instructions or computation. Traditional CPython threads usually do not make this faster because only one thread executes Python bytecode at a time. Thread scheduling and lock handoffs may add overhead.
- **I/O-bound:** spends time waiting for files, sockets, databases, or other external systems. Threads can make progress while another thread is blocked; many I/O operations release the GIL while waiting. `asyncio` can achieve similar overlap with cooperative tasks and fewer threads.

### Thread switching

The interpreter periodically checks whether another thread should run. Switching is implementation- and workload-dependent, and a context switch can occur between bytecode instructions. Never rely on a particular switch point or on an operation merely appearing short. Use locks, queues, events, and other synchronization primitives around shared invariants.

In [1]:
import os
import sys
import threading
import time

print("implementation:", sys.implementation.name)
print("Python version:", sys.version.split()[0])
print("CPU count:", os.cpu_count())
print("GIL enabled in this build:", sys._is_gil_enabled())


def cpu_work(iterations):
    total = 0
    for number in range(iterations):
        total += number * number
    return total


def timed_call(function, *args):
    started = time.perf_counter()
    result = function(*args)
    return result, time.perf_counter() - started


iterations = 300_000
_, serial_cpu_time = timed_call(cpu_work, iterations * 2)

started = time.perf_counter()
threads = [threading.Thread(target=cpu_work, args=(iterations,)) for _ in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
thread_cpu_time = time.perf_counter() - started

print(f"CPU serial time: {serial_cpu_time:.4f}s")
print(f"CPU two-thread time: {thread_cpu_time:.4f}s")


def wait_briefly(label):
    time.sleep(0.15)
    return label

started = time.perf_counter()
serial_io = [wait_briefly(label) for label in ("A", "B")]
serial_io_time = time.perf_counter() - started

started = time.perf_counter()
results = []


def worker(label):
    results.append(wait_briefly(label))


io_threads = [threading.Thread(target=worker, args=(label,)) for label in ("A", "B")]
for thread in io_threads:
    thread.start()
for thread in io_threads:
    thread.join()
io_thread_time = time.perf_counter() - started

print("I/O serial results:", serial_io)
print("I/O threaded results:", sorted(results))
print(f"I/O serial time: {serial_io_time:.4f}s")
print(f"I/O two-thread time: {io_thread_time:.4f}s")

implementation: cpython
Python version: 3.13.0
CPU count: 8
GIL enabled in this build: True
CPU serial time: 0.0426s
CPU two-thread time: 0.0414s
I/O serial results: ['A', 'B']
I/O threaded results: ['A', 'B']
I/O serial time: 0.3015s
I/O two-thread time: 0.1518s


## 2. Race conditions and synchronization

A **race condition** occurs when the result depends on the timing of concurrent operations. The GIL does not make a multi-step operation atomic. For example, incrementing a shared counter involves reading, computing, and writing; another thread can interfere between those steps.

Use `threading.Lock` when multiple threads must maintain an invariant over shared mutable state. Other tools include `RLock`, `Semaphore`, `Event`, `Condition`, `Barrier`, `queue.Queue`, and thread-safe design that avoids shared mutation. Keep critical sections small, and acquire multiple locks in a consistent order to reduce deadlock risk.

A lock protects a particular section only when all participating code uses the same lock. It does not magically protect the object everywhere else.

In [2]:
import threading


def increment_without_lock(state, repeats):
    for _ in range(repeats):
        current = state["value"]
        time.sleep(0)
        state["value"] = current + 1


def increment_with_lock(state, repeats, lock):
    for _ in range(repeats):
        with lock:
            state["value"] += 1


repeats = 100
unsafe_state = {"value": 0}
unsafe_threads = [threading.Thread(target=increment_without_lock, args=(unsafe_state, repeats)) for _ in range(4)]
for thread in unsafe_threads:
    thread.start()
for thread in unsafe_threads:
    thread.join()
print("unsafe counter:", unsafe_state["value"], "expected:", repeats * 4)

safe_state = {"value": 0}
lock = threading.Lock()
safe_threads = [threading.Thread(target=increment_with_lock, args=(safe_state, repeats, lock)) for _ in range(4)]
for thread in safe_threads:
    thread.start()
for thread in safe_threads:
    thread.join()
print("safe counter:", safe_state["value"], "expected:", repeats * 4)

unsafe counter: 280 expected: 400
safe counter: 400 expected: 400


## 3. Multiprocessing and AsyncIO

### Multiprocessing

`multiprocessing` creates separate processes, each with its own interpreter, memory space, and usually its own GIL. Separate processes can execute CPU-bound Python code in parallel across cores, at the cost of process startup, serialization, memory, and inter-process communication. Use pools or executors for independent CPU-heavy tasks. Values must cross process boundaries explicitly through pipes, queues, shared memory, or serialized arguments/results.

### AsyncIO

`asyncio` uses cooperative concurrency. An event loop runs tasks until they reach an `await`, then schedules another ready task. It is excellent for many I/O-bound operations when the libraries involved are non-blocking and provide async APIs. AsyncIO does not make CPU-bound code parallel: a long synchronous computation blocks the event loop. Move CPU-heavy work to a process pool or use another appropriate strategy.

Threads are preemptively scheduled by the runtime/OS and are convenient for blocking synchronous APIs. Async tasks yield cooperatively and can be cheaper in large numbers, but every operation in the event-loop thread must avoid blocking.

In [6]:
import asyncio
import subprocess


process_code = """
import os
import sys

number = int(sys.argv[1])
result = sum(value * value for value in range(number))
print(f'pid={os.getpid()}, result={result}')
"""

started = time.perf_counter()
processes = [
    subprocess.Popen(
        [sys.executable, "-c", process_code, "100000"],
        stdout=subprocess.PIPE,
        text=True,
    )
    for _ in range(2)
]
process_outputs = [process.communicate()[0].strip() for process in processes]
process_time = time.perf_counter() - started
print("separate process results:", process_outputs)
print(f"two-process elapsed time: {process_time:.4f}s")


async def async_wait(label):
    await asyncio.sleep(0.15)
    return label


async def run_async_work():
    started = time.perf_counter()
    results = await asyncio.gather(async_wait("A"), async_wait("B"))
    return results, time.perf_counter() - started


async_results, async_time = await run_async_work()
print("async results:", async_results)
print(f"asyncio elapsed time: {async_time:.4f}s")

separate process results: ['pid=19712, result=333328333350000', 'pid=14796, result=333328333350000']
two-process elapsed time: 0.1269s
async results: ['A', 'B']
asyncio elapsed time: 0.1521s


## 4. Modern free-threaded Python

The GIL is no longer the only CPython direction. Recent CPython releases provide experimental or optional **free-threaded builds** (often called no-GIL builds) in which multiple threads can execute Python code in parallel. The goal is better CPU-bound thread scaling while preserving Python's usability and much of its compatibility.

Free-threaded Python has tradeoffs:

- C extensions must be audited and adapted for the new concurrency model.
- Some single-threaded workloads may have overhead from stronger synchronization or changed object internals.
- Thread safety still belongs to application code; removing the GIL does not make shared mutable state logically safe.
- Builds, wheels, and library support may differ from the standard GIL-enabled build.
- Correct code should not depend on the GIL as an accidental lock.

Check the active build with:

```python
import sys
sys._is_gil_enabled()
```

A result of `True` means the traditional GIL is enabled for that interpreter. A free-threaded build may return `False`. Treat this as an implementation-specific diagnostic, not a portable language guarantee.

### Decision guide

| Workload | Good first choice | Reason |
|---|---|---|
| Blocking synchronous I/O | `threading` | Overlap waits while keeping familiar APIs |
| Many non-blocking I/O tasks | `asyncio` | Cooperative scheduling with low task overhead |
| CPU-heavy pure Python on GIL builds | `multiprocessing` or processes | Separate interpreters can run on multiple cores |
| CPU-heavy native code that releases the GIL | Threads may help | Work executes outside the Python bytecode bottleneck |
| CPU-heavy code on supported free-threaded CPython | Threads may help | Python threads can execute in parallel, subject to library support |
| Shared state with complex coordination | Locks, queues, or message passing | Make ownership and synchronization explicit |

## 5. Practice lab

Attempt these exercises before consulting a solution.

1. Repeat the CPU-bound benchmark with different iteration counts. Explain why a small benchmark can be dominated by startup and scheduling overhead.
2. Replace `time.sleep` with a blocking file or network operation and explain why threads can overlap waiting.
3. Modify the unsafe counter so the race is more likely to appear, then protect it with a `Lock`.
4. Use a `queue.Queue` to send work from producer threads to a consumer thread without sharing a mutable list directly.
5. Write a CPU task and run it serially, with threads, and with separate processes. Record wall-clock time and explain the result on a GIL-enabled build.
6. Convert the I/O thread example to `asyncio.gather`. Identify every `await` point where another task can run.
7. Add a blocking `time.sleep` inside an async coroutine. Observe how it changes all task timing, then replace it with `await asyncio.sleep`.
8. Design a function that is safe both on traditional GIL-enabled CPython and on a free-threaded build. Identify its shared-state invariant and synchronization strategy.
9. Explain why “the GIL makes Python thread-safe” is incorrect. Give one example of a protected invariant.
10. Check `sys._is_gil_enabled()` and write down how your benchmark conclusions would change if it returned `False`.

### Review checklist

- [ ] I distinguish concurrency from parallelism.
- [ ] I know what the traditional CPython GIL protects and what it does not protect.
- [ ] I can identify CPU-bound and I/O-bound workloads.
- [ ] I understand why thread switching can expose races in multi-step operations.
- [ ] I use locks or message passing for shared invariants.
- [ ] I choose processes for CPU-bound pure Python on GIL-enabled builds.
- [ ] I choose threads for blocking I/O when synchronous APIs are appropriate.
- [ ] I use `asyncio` only with non-blocking async operations.
- [ ] I know a free-threaded build changes scaling possibilities, not the need for correct synchronization.

### Final mental model

```text
threads: shared memory, convenient, useful for I/O, synchronize shared state
processes: separate memory, true CPU parallelism, serialization/IPC cost
asyncio: cooperative tasks, explicit await points, excellent for non-blocking I/O
free-threaded CPython: optional parallel Python execution, ecosystem and locking tradeoffs
```